# s00a_tone_fft_intro
This notebook started as a line-by-line conversion of the example script `docs/examples_tutorial/e00_intro_set/s00a_tone_fft_intro.py`.



## Introduction to Time-Frequency Representations (TFRs)

The foundation of efficient TFR computation is the Fast Fourier Transform (FFT), the clever child of the Discrete Fouriet Transform. For N = number of points in a digital signal, computation scales as N log(N) instead of N**2. The FFT works exceptionally well when N is a power of two, and FFT algorithms have on-board methods to pad 'short' signals with zeroes to construct dyadic durations. If unplanned for, this can lead to ringing at the edges.

This first case study constructs a sinusoid input signal with unit amplitude to validate that the nominal FFT power averaged over the signal duration is 1/2.

Only numpy is imported. If the use case expects absolute (physical) units as an output, it is recommended that each FFT algorithm is tested and evaluated before extensive usage to verify results are as expected.

In [ ]:
import numpy as np

Automatically created module for IPython interactive environment


### Monotone design

The snippet explains assumptions and practical constraints when working with discrete-time sinusoids and spectra.

An evenly sampled input is assumed, where the sample rate is a nominal constant measured in samples per second (sps or Hz). Even sampling is important because almost all frequency-domain analysis and synthesis formulas assume uniform spacing. If the acquisition clock "wobbles" or samples arrive irregularly, resampling or interpolation can be used to reconstruct a uniformly sampled signal before spectral processing.

The highest representable frequency in the digital domain is the Nyquist frequency, which is one half of the sample rate. This is expressed as $$f_N = \frac{f_s}{2}$$ and means all meaningful spectral content lies in $[0, f_N]$. Any energy above $f_N$ will fold back (alias) into this band and corrupt the measured spectrum.

To avoid aliasing, the analog front-end should include an anti-aliasing low-pass filter that attenuates signal power above the passband so that residual energy near Nyquist is negligible compared to noise. Designing that filter aggressively (steep transition, sufficient stopband attenuation) is a key practical requirement—otherwise a digitally generated or analyzed sinusoid may show spurious components caused by aliasing rather than the true signal.

Finally, the example sets a low sample rate of $f_s=800\ \text{Hz}$, which gives a Nyquist frequency of $f_N=400\ \text{Hz}$. That defines the usable spectral canvas: frequencies of interest should be below 400 Hz or be band-limited and filtered accordingly before sampling.

In [ ]:
# Parameters and derived quantities
frequency_sample_rate_hz = 800.
frequency_center_hz = 60.
time_duration_s = 1  # Nominal value
# Use the whole record to compute the fft
time_fft_s = time_duration_s
frequency_resolution_hz = 1 / time_fft_s


In [ ]:

# Make durations a power of two (FFT-friendly)
time_duration_nd = 2 ** (int(np.log2(time_duration_s * frequency_sample_rate_hz)))
time_fft_nd = 2 ** (int(np.log2(time_fft_s * frequency_sample_rate_hz)))

# Positive FFT frequencies and locate the closest FFT bin to the nominal center frequency
frequency_fft_pos_hz = np.fft.rfftfreq(time_fft_nd, d=1/frequency_sample_rate_hz)
fft_index = np.argmin(np.abs(frequency_fft_pos_hz - frequency_center_hz))
frequency_center_fft_hz = frequency_fft_pos_hz[fft_index]
frequency_resolution_fft_hz = frequency_sample_rate_hz / time_fft_nd


In [5]:
# Print diagnostic information
print('Nyquist frequency:', frequency_sample_rate_hz / 2)
print('Nominal signal frequency, hz:', frequency_center_hz)
print('FFT signal frequency, hz:', frequency_center_fft_hz)
print('Nominal spectral resolution, hz', frequency_resolution_hz)
print('FFT spectral resolution, hz', frequency_resolution_fft_hz)
print('Number of FFT points:', time_duration_nd)
print('log2(FFT points):', np.log2(time_duration_nd))

Nyquist frequency: 400.0
Nominal signal frequency, hz: 60.0
FFT signal frequency, hz: 59.375
Nominal spectral resolution, hz 1.0
FFT spectral resolution, hz 1.5625
Number of FFT points: 512
log2(FFT points): 9.0


In [ ]:

# Dimensionless conversions and time vector (samples)
frequency_center = frequency_center_hz / frequency_sample_rate_hz
frequency_center_fft = frequency_center_fft_hz / frequency_sample_rate_hz
time_nd = np.arange(time_duration_nd)

# Construct the synthetic tone (maximum FFT amplitude at exact FFT frequency)
mic_sig = np.cos(2 * np.pi * frequency_center_fft * time_nd)

In [ ]:
# Compute the RFFT of the whole record and compare to the full FFT
fft_sig_pos = np.fft.rfft(mic_sig)
fft_sig = np.fft.fft(mic_sig)
print('RFFT returns only the positive frequencies')
print('len(FFT):', len(fft_sig))
print('len(RFFT):', len(fft_sig_pos))
print('RFFT[fc]:', fft_sig_pos[fft_index])

# Scale and compute power
fft_abs_pos_over_N = np.abs(fft_sig_pos) / len(mic_sig)
fft_abs_power = 2 * fft_abs_pos_over_N ** 2



Nyquist frequency: 400.0
Nominal signal frequency, hz: 60.0
FFT signal frequency, hz: 59.375
Nominal spectral resolution, hz 1.0
FFT spectral resolution, hz 1.5625
Number of FFT points: 512
log2(FFT points): 9.0
RFFT returns only the positive frequencies
len(FFT): 512
len(RFFT): 257
RFFT[fc]: (256-3.03391770985856e-12j)


### Check amplitudes

In [ ]:
print('|RFFT(fc) / N|:', fft_abs_pos_over_N[fft_index])
print('2 * |RFFT(fc) / N|**2:', fft_abs_power[fft_index])
print('*** SUMMARY: FFT of a constant frequency tone with unit peak amplitude ***')
print('Positive frequency FFT amplitude is 1/2, negative frequency FFT amplitude is 1/2')
print('Power averaged over the signal duration is P**2 = 2 * |RFFT / N|**2 = 1/2')
print('RMS amplitude is sqrt(P**2) = 1 / sqrt(2)')
print('** IMPORTANT NOTE: EXACT RECONSTRUCTION ONLY OCCURS AT FFT FREQUENCY **')

In [ ]:
import matplotlib.pyplot as plt
# Plot waveform and FFT power
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, constrained_layout=True, figsize=(8, 5))
ax1.plot(time_nd / frequency_sample_rate_hz, mic_sig)
ax1.set_title('Synthetic tone, no taper')
ax1.set_xlabel('Time, s')
ax1.set_ylabel('Norm')
ax2.semilogx(frequency_fft_pos_hz, fft_abs_power)
ax2.set_title(f"FFT Power, f = {frequency_center_fft_hz:.3f} Hz")
ax2.set_xlabel('Frequency, Hz')
ax2.set_ylabel('$2\cdot\mid\frac{RFFT}{N}\mid ^2$')
ax2.grid(True)
plt.show()

## Notes

- Exact reconstruction occurs only when the sinusoid frequency aligns with an FFT bin.
- If you change `frequency_center_hz` to a non-bin frequency, the peak amplitude will spread across neighbouring bins (spectral leakage).
- Try increasing `time_fft_s` (making longer records or padding zeros) to increase spectral resolution.